Source for this notebook: https://github.com/openai/openai-cookbook/blob/main/examples/gpt-5/gpt-5_new_params_and_tools.ipynb

> **Model update (GPT-5.6 refresh).** All `model=` calls in this notebook now target **`gpt-5.6-sol`** (the bare `gpt-5.6` alias also routes to Sol), the current flagship model for coding and professional work. `reasoning.effort` is pinned explicitly in cells that care about cost/latency; GPT-5.6 defaults to `medium`.
>
> This notebook uses `gpt-5.6-sol` throughout for consistency.

#  GPT-5 New Params and Tools

We’re introducing new developer controls in the GPT-5 series that give you greater control over model responses—from shaping output length and style to enforcing strict formatting. Below is a quick overview of the latest features:


| #  | Feature | Overview | Values / Usage |
|----|---------|----------|----------------|
| 1. | **Verbosity Parameter** | Lets you hint the model to be more or less expansive in its replies. Keep prompts stable and use the parameter instead of re-writing. | • **low** → terse UX, minimal prose.<br>• **medium** *(default)* → balanced detail.<br>• **high** → verbose, great for audits, teaching, or hand-offs. |
| 2. | **Freeform Function Calling** | Generate raw text payloads—anything from Python scripts to SQL queries—directly to your custom tool without JSON wrapping. Offers greater flexibility for external runtimes like:<br>• Code sandboxes (Python, C++, Java, …)<br>• SQL databases<br>• Shell environments<br>• Config generators | Use when structured JSON isn’t needed and raw text is more natural for the target tool. |
| 3. | **Context-Free Grammar (CFG)** | A set of production rules defining valid strings in a language. Each rule rewrites a non-terminal into terminals and/or other non-terminals, independent of surrounding context. Useful for constraining output to match the syntax of programming languages or custom formats in OpenAI tools. | Use as a contract to ensure the model emits only valid strings accepted by the grammar. |
| 4. | **Low-Effort Reasoning** | Runs the model with minimal reasoning tokens to minimize latency and speed time-to-first-token. Ideal for deterministic, lightweight tasks (extraction, formatting, short rewrites, simple classification) where explanations aren’t needed. If not specified, effort defaults to medium. Note: `minimal` was available at the original GPT-5 launch but is **not** supported on any GPT-5.6 tier (Sol/Terra/Luna) — use `low` instead. | Set reasoning effort: `"low"`. Avoid for multi-step planning or tool-heavy workflows. |


**Supported Models:**  
- gpt-5.6-sol (alias `gpt-5.6`)  
- gpt-5.6-terra  
- gpt-5.6-luna  

**Supported API Endpoints** 
- Responses API 
- Chat Completions API 

Note: We recommend to use Responses API with GPT-5 series of model to get the most performance out of the models. 


## Prerequisites 

Let's begin with updating your OpenAI SDK that supports the new params and tools for GPT-5. Make sure you've set OPENAI_API_KEY as an environment variable. 

In [1]:
import os
import getpass

def _set_env(var: str):
    if not os.environ.get(var):
        os.environ[var] = getpass.getpass(f"var: ")

_set_env("OPENAI_API_KEY")

In [2]:
# Confirm installed package versions (importlib avoids PATH/shell issues with pip)
import importlib.metadata as _im
print("openai", _im.version("openai"))
print("pandas", _im.version("pandas"))

openai 2.48.0
pandas 2.3.2


## 1. Verbosity Parameter 

### 1.1 Overview 
The verbosity parameter lets you hint the model to be more or less expansive in its replies.   

**Values:** "low", "medium", "high"

- low → terse UX, minimal prose.
- medium (default) → balanced detail.
- high → verbose, great for audits, teaching, or hand-offs.

Keep prompts stable and use the param rather than re-writing.


In [3]:
from openai import OpenAI
import pandas as pd
from IPython.display import display

client = OpenAI()

question = "Write a poem about a boy and his first pet dog."

data = []

for verbosity in ["low", "medium", "high"]:
    response = client.responses.create(
        model="gpt-5.6-sol",
        input=question,
        text={"verbosity": verbosity},
        reasoning={"effort": "low"}
    )

    # Extract text
    output_text = ""
    for item in response.output:
        if hasattr(item, "content") and type(item.content)==list:
            for content in item.content:
                if hasattr(content, "text"):
                    output_text += content.text

    usage = response.usage
    data.append({
        "Verbosity": verbosity,
        "Sample Output": output_text,
        "Output Tokens": usage.output_tokens
    })

# Create DataFrame
df = pd.DataFrame(data)

# Display nicely with centered headers
pd.set_option('display.max_colwidth', None)
styled_df = df.style.set_table_styles(
    [
        {'selector': 'th', 'props': [('text-align', 'center')]},  # Center column headers
        {'selector': 'td', 'props': [('text-align', 'left')]}     # Left-align table cells
    ]
)

display(styled_df)

,Verbosity,Sample Output,Output Tokens
0,low,"A boy held out his trembling hand, The puppy sniffed, then took his stand— A wagging tail, a tiny bark, Two new friends beneath the dark. They raced through fields and splashed through rain, Shared secret joys and little pains. And every night, the boy would say, “Tomorrow brings another play.” His first true friend on four small feet, With muddy paws and love complete. A boy, a dog, a bond begun— Two faithful hearts that beat as one.",113
1,medium,"The day he brought the puppy home, The sky was soft and blue; A wagging tail, two clumsy paws, A friendship bright and new. He named him Scout and shared his bed, His secrets and his fears; They chased the wind through summer fields And splashed through autumn years. The boy threw sticks; the dog brought back A treasure, proud and grand. And when the world felt far too big, Scout gently licked his hand. Through muddy tracks and midnight barks, They learned what love could be: A boy, his first and faithful dog— Two hearts running free.",151
2,high,"### **The Boy and the Dog** On a bright and windy Saturday, When clouds went sailing by, A boy stood at a garden gate With wonder in his eyes. For there, beneath the apple tree, A puppy small and brown Was tumbling through the clover leaves, His ears flopped upside down. The puppy sniffed the boy’s old shoes, Then licked his muddy knee, And in that moment both of them Knew they were meant to be. The boy called him **Comet**, For the blaze upon his chest, And promised him a blanket warm, A bowl, a bone, and rest. They chased the sun through summer fields And splashed in autumn rain; They built a fort, explored the woods, Then hurried home again. At night, when thunder shook the glass And shadows crossed the wall, The boy would whisper, “I’m right here,” Though he felt scared as well. And Comet laid his gentle head Against the boy’s small hand— No words were needed between friends Who seemed to understand. The boy taught Comet how to sit, To fetch, to stay, to run; But Comet taught the boy still more— That love could just be fun. It lived in every wagging tail, Each footprint by the door, In waiting through the school-day hours And racing ’cross the floor. And years from then, the boy would keep, Wherever he might roam, The memory of his first true friend— The dog who made home **home**.",358


In [4]:
# LIVE token-count benchmark across verbosity levels (replaces hardcoded 560/849/1288).
# Re-run this with a valid API key to get real numbers for your model/prompt.
import pandas as pd
from openai import OpenAI
client = OpenAI()

bench_question = "Write a poem about a boy and his first pet dog."
rows = []
for verbosity in ["low", "medium", "high"]:
    r = client.responses.create(
        model="gpt-5.6-sol",
        input=bench_question,
        text={"verbosity": verbosity},
        reasoning={"effort": "low"},   # pinned so verbosity is the only variable
    )
    rows.append({"Verbosity": verbosity, "Output Tokens": r.usage.output_tokens})

bench_df = pd.DataFrame(rows)
print(bench_df.to_string(index=False))
print("\nLow -> Medium -> High output tokens:",
      " -> ".join(str(x) for x in bench_df["Output Tokens"].tolist()))

Verbosity  Output Tokens
      low            124
   medium            205
     high            285

Low -> Medium -> High output tokens: 124 -> 205 -> 285


> **Note:** The benchmark cell above runs live against `gpt-5.6-sol`. Re-run it to get current token counts; the exact numbers vary by model generation but the trend (low < medium < high) is stable.

### 2.3 Using Verbosity for Coding Use Cases 

The verbosity parameter also influences the length and complexity of generated code, as well as the depth of accompanying explanations. Here's an example, wherein we use various verboisty levels for a task to generate a Python program that sorts an array of 1000000 random numbers. 

In [5]:
from openai import OpenAI

client = OpenAI()

prompt = "Write a Python function to reverse a string"

def ask_with_verbosity(verbosity: str, question: str):
    response = client.responses.create(
        model="gpt-5.6-sol",
        input=question,
        text={
            "verbosity": verbosity
        }
    )

    # Extract assistant's text output
    output_text = ""
    for item in response.output:
        if hasattr(item, "content") and type(item.content)==list:
            for content_item in item.content:
                if hasattr(content_item, "text"):
                    output_text += content_item.text

    # Token usage details
    usage = response.usage

    print("--------------------------------")
    print(f"Verbosity: {verbosity}")
    print("Output:")
    print(output_text)
    print("Tokens => input: {} | output: {}".format(
        usage.input_tokens, usage.output_tokens
    ))


# Example usage:
ask_with_verbosity("low", prompt)

--------------------------------
Verbosity: low
Output:
```python
def reverse_string(text: str) -> str:
    return text[::-1]
```
Tokens => input: 14 | output: 24


Notice that the code output is a plain script. Now, lets run with 'medium' 

In [6]:
ask_with_verbosity("medium", prompt)

--------------------------------
Verbosity: medium
Output:
```python
def reverse_string(text: str) -> str:
    return text[::-1]
```

Example:

```python
print(reverse_string("hello"))  # "olleh"
```
Tokens => input: 14 | output: 43


Medium verboisty, generated richer code with additioanl explanations. Let's do the same with high. 

In [7]:
ask_with_verbosity("high", prompt)

--------------------------------
Verbosity: high
Output:
```python
def reverse_string(text: str) -> str:
    """Return the given string in reverse order."""
    return text[::-1]
```

Example:

```python
print(reverse_string("Hello, world!"))
# Output: !dlrow ,olleH
```
Tokens => input: 14 | output: 60


High verbosity yielded additional details and explanations. 

### 1.3 Takeaways 

The new verbosity parameter reliably scales both the length and depth of the model’s output while preserving correctness and reasoning quality - **without changing the underlying prompt**.
In this example:

- **Low verbosity** produces a minimal, functional script with no extra comments or structure.
- **Medium verbosity** adds explanatory comments, function structure, and reproducibility controls.
- **High verbosity** yields a comprehensive, production-ready script with argument parsing, multiple sorting methods, timing/verification, usage notes, and best-practice tips.

## 2. Free‑Form Function Calling

### 2.1 Overview 
GPT‑5 can now send raw text payloads - anything from Python scripts to SQL queries - to your custom tool without wrapping the data in JSON using the new tool `"type": "custom"`. This differs from classic structured function calls, giving you greater flexibility when interacting with external runtimes such as:

- code_exec with sandboxes (Python, C++, Java, …)
- SQL databases
- Shell environments
- Configuration generators

**Note that custom tool type does NOT support parallel tool calling.**

### 2.2 Quick Start Example - Compute the Area of a Circle

The code below produces a simple python code to calculate area of a circle, and instruct the model to use the freeform tool call to output the result. 

In [8]:
from openai import OpenAI

client = OpenAI()

response = client.responses.create(
    model="gpt-5.6-sol",
    input="Please use the code_exec tool to calculate the area of a circle with radius equal to the number of 'r's in strawberry",
    text={"format": {"type": "text"}},
    tools=[
        {
            "type": "custom",
            "name": "code_exec",
            "description": "Executes arbitrary python code",
        }
    ]
)
print(response.output)

[ResponseReasoningItem(id='rs_05ced7cd8dfd35aa006a6674595e20819c93c1a19b0a703860', summary=[], type='reasoning', content=[], encrypted_content='gAAAAABqZnRaQqov_16Knpm2z_CElsxQhFQ4hGepqLG9SbWxrYCCJHPW7-mDtq6e9ebSa3wdYnzg7YV3UcqTj374s1kx6eQjKduYB7EAaGCms9xViJNLlVZ9MaTV-n001-YojYUWfmhCavUa9OdIs79w49rzNhokqdT_Vczy3AdDvwwNSEMxSbcQ0mAnyONAuNAJVf8F8IbbVEBMrF3At_9mCtKxJkH9I0dRpRZ9_m06Q6UT3M0gumC21YKj9a6ImuYNvnDlZyD2kpj59Dfqmxxljqn1Nb-O8Q8gF4jCQyHcg4iGknZjNK4AbTW91-6-MW37_q7Wz0-v0oujjBhJhiGK7ifNCiGq2jJUxWLAXWnAA6TbQlvqBi4afYdI_8sefYy9pp9Aia0c3HD1k8UAvO0i1pdvqr_GmAhNaPB1pwQXATG_BNRPjN8E4iD_jCbATG-P3D6NPOUlZRfmCX6ZH08QXdGNM4hIcE3CjhdWTBXFCcQYSRam-_DOlxq4lJZ7YP-yL31BxW_nZ1rWQ3mltu87GqgA7ky8-3hCxcc_JGp3eB8TKXCR_5g5pfg7ZD2TJhdA-8Dph_F_wklUKhWxMifdQhFXNR6uhDHt17RMp3Gul66lGd2CLE_2Q16TyvlWXa42h04vbEZVsyXof8jNztMt3bC7kBW3pf4_CVwDyEBQEqJjtIqSbC2tuoSWKi2n7nvPtEMEq2IOPBGBVLepf0SSK4wqO1KRlSpN-fCTmG4QRuL52MjLpL4sygvEO5X3TuJcVxeCN-UoP92BPIDuVpBbryaG5pMUwrLnZb0ujwwzXG2Hvkl0wfyR5HSmJzJDIL7BWJnh7fJe51vGjW2jFysu

The model emits a `tool call` containing raw Python. You execute that code server‑side, capture the printed result, and send it back in a follow‑up responses.create call.

### 2.3 Mini‑Benchmark – Sorting an Array in Three Languages
To illustrate the use of free form tool calling, we will ask GPT‑5 to:
- Generate Python, C++, and Java code that sorts a fixed array 10 times.
- Print only the time (in ms) taken for each iteration in the code. 
- Call all three functions, and then stop 

In [9]:
from openai import OpenAI
from typing import List, Optional

MODEL_NAME = "gpt-5.6-sol"

# Tools that will be passed to every model invocation. They are defined once so
# that the configuration lives in a single place.
TOOLS = [
    {
        "type": "custom",
        "name": "code_exec_python",
        "description": "Executes python code",
    },
    {
        "type": "custom",
        "name": "code_exec_cpp",
        "description": "Executes c++ code",
    },
    {
        "type": "custom",
        "name": "code_exec_java",
        "description": "Executes java code",
    },
]

client = OpenAI()

def create_response(
    input_messages: List[dict],
    previous_response_id: Optional[str] = None,
):
    """Wrapper around ``client.responses.create``.

    Parameters
    ----------
    input_messages: List[dict]
        The running conversation history to feed to the model.
    previous_response_id: str | None
        Pass the ``response.id`` from the *previous* call so the model can keep
        the thread of the conversation.  Omit on the very first request.
    """
    kwargs = {
        "model": MODEL_NAME,
        "input": input_messages,
        "text": {"format": {"type": "text"}},
        "tools": TOOLS,
    }
    if previous_response_id:
        kwargs["previous_response_id"] = previous_response_id

    return client.responses.create(**kwargs)

# Recursive 
def run_conversation(
    input_messages: List[dict],
    previous_response_id: Optional[str] = None,
):
  
    response = create_response(input_messages, previous_response_id)

    # ``response.output`` is expected to be a list where element 0 is the model
    # message.  Element 1 (if present) denotes a tool call.  When the model is
    # done with tool calls, that element is omitted.
    tool_call = response.output[1] if len(response.output) > 1 else None

    if tool_call and tool_call.type == "custom_tool_call":
        print("--- tool name ---")
        print(tool_call.name)
        print("--- tool call argument (generated code) ---")
        print(tool_call.input)
        
        # Add a synthetic *tool result* so the model can continue the thread.
        
        input_messages.append(
            {
                "type": "function_call_output",
                "call_id": tool_call.call_id,
                "output": "done", # <-- replace with the result of the tool call
            }
        )

        # Recurse with updated conversation and track the response id so the
        # model is aware of the prior turn.
        return run_conversation(input_messages, previous_response_id=response.id)
    else:
        # Base-case: no further tool call - return. 
        return 


prompt = """
Write code to sort the array of numbers in three languages: C++, Python and Java (10 times each)using code_exec functions.

ALWAYS CALL THESE THREE FUNCTIONS EXACTLY ONCE: code_exec_python, code_exec_cpp and code_exec_java tools to sort the array in each language. Stop once you've called these three functions in each language once.

Print only the time it takes to sort the array in milliseconds. 

[448, 986, 255, 884, 632, 623, 246, 439, 936, 925, 644, 159, 777, 986, 706, 723, 534, 862, 195, 686, 846, 880, 970, 276, 613, 736, 329, 622, 870, 284, 945, 708, 267, 327, 678, 807, 687, 890, 907, 645, 364, 333, 385, 262, 730, 603, 945, 358, 923, 930, 761, 504, 870, 561, 517, 928, 994, 949, 233, 137, 670, 555, 149, 870, 997, 809, 180, 498, 914, 508, 411, 378, 394, 368, 766, 486, 757, 319, 338, 159, 585, 934, 654, 194, 542, 188, 934, 163, 889, 736, 792, 737, 667, 772, 198, 971, 459, 402, 989, 949]
"""

# Initial developer message.
messages = [
    {
        "role": "developer",
        "content": prompt,
    }
]

run_conversation(messages)


--- tool name ---
code_exec_python
--- tool call argument (generated code) ---
import time

numbers = [448, 986, 255, 884, 632, 623, 246, 439, 936, 925, 644, 159, 777, 986, 706, 723, 534, 862, 195, 686, 846, 880, 970, 276, 613, 736, 329, 622, 870, 284, 945, 708, 267, 327, 678, 807, 687, 890, 907, 645, 364, 333, 385, 262, 730, 603, 945, 358, 923, 930, 761, 504, 870, 561, 517, 928, 994, 949, 233, 137, 670, 555, 149, 870, 997, 809, 180, 498, 914, 508, 411, 378, 394, 368, 766, 486, 757, 319, 338, 159, 585, 934, 654, 194, 542, 188, 934, 163, 889, 736, 792, 737, 667, 772, 198, 971, 459, 402, 989, 949]

start = time.perf_counter_ns()
for _ in range(10):
    sorted_numbers = sorted(numbers)
elapsed_ms = (time.perf_counter_ns() - start) / 1_000_000
print(f"{elapsed_ms:.6f}")



--- tool name ---
code_exec_cpp
--- tool call argument (generated code) ---
#include <algorithm>
#include <chrono>
#include <iomanip>
#include <iostream>
#include <vector>

int main() {
    const std::vector<int> numbers = {
        448, 986, 255, 884, 632, 623, 246, 439, 936, 925, 644, 159, 777, 986, 706, 723, 534, 862, 195, 686,
        846, 880, 970, 276, 613, 736, 329, 622, 870, 284, 945, 708, 267, 327, 678, 807, 687, 890, 907, 645,
        364, 333, 385, 262, 730, 603, 945, 358, 923, 930, 761, 504, 870, 561, 517, 928, 994, 949, 233, 137,
        670, 555, 149, 870, 997, 809, 180, 498, 914, 508, 411, 378, 394, 368, 766, 486, 757, 319, 338, 159,
        585, 934, 654, 194, 542, 188, 934, 163, 889, 736, 792, 737, 667, 772, 198, 971, 459, 402, 989, 949
    };

    volatile long long checksum = 0;
    const auto start = std::chrono::steady_clock::now();
    for (int i = 0; i < 10; ++i) {
        std::vector<int> sortedNumbers = numbers;
        std::sort(sortedNumbers.begin(), sortedNu

--- tool name ---
code_exec_java
--- tool call argument (generated code) ---
import java.util.Arrays;

public class Main {
    public static void main(String[] args) {
        int[] numbers = {
            448, 986, 255, 884, 632, 623, 246, 439, 936, 925, 644, 159, 777, 986, 706, 723, 534, 862, 195, 686,
            846, 880, 970, 276, 613, 736, 329, 622, 870, 284, 945, 708, 267, 327, 678, 807, 687, 890, 907, 645,
            364, 333, 385, 262, 730, 603, 945, 358, 923, 930, 761, 504, 870, 561, 517, 928, 994, 949, 233, 137,
            670, 555, 149, 870, 997, 809, 180, 498, 914, 508, 411, 378, 394, 368, 766, 486, 757, 319, 338, 159,
            585, 934, 654, 194, 542, 188, 934, 163, 889, 736, 792, 737, 667, 772, 198, 971, 459, 402, 989, 949
        };

        long checksum = 0;
        long start = System.nanoTime();
        for (int i = 0; i < 10; i++) {
            int[] sortedNumbers = numbers.clone();
            Arrays.sort(sortedNumbers);
            checksum += sortedNumbers[

The model output three code blocks in Python, C++ and Java for the same algorithm. The output of the function call was chained back into the model as input to allow model to keep going until all the functions have been called exactly once. 

### 2.4 Takeaways 

Freeform tool calling in GPT-5 lets you send raw text payloads—such as Python scripts, SQL queries, or config files—directly to custom tools without JSON wrapping. This provides greater flexibility for interacting with external runtimes and allows the model to generate code or text in the exact format your tool expects. It’s ideal when structured JSON is unnecessary and natural text output improves usability.

> **See also:** for OpenAI's own server-side alternative to a hand-written tool-calling loop, see the "Programmatic Tool Calling" callout in `6.0-live-demo-function-calling.ipynb`.

## 3. Context‑Free Grammar (CFG)

### 3.1 Overview 
A context‑free grammar is a collection of production rules that define which strings belong to a language. Each rule rewrites a non‑terminal symbol into a sequence of terminals (literal tokens) and/or other non‑terminals, independent of surrounding context—hence context‑free. CFGs can capture the syntax of most programming languages and, in OpenAI custom tools, serve as contracts that force the model to emit only strings that the grammar accepts.

### 3.2 Grammar Fundamentals

**Supported Grammar Syntax** 
- Lark - https://lark-parser.readthedocs.io/en/stable/
- Regex - https://docs.rs/regex/latest/regex/#syntax

We use LLGuidance under the hood to constrain model sampling: https://github.com/guidance-ai/llguidance.

**Unsupported Lark Features** 
- Lookaround in regexes (`(?=...)`, `(?!...)`, etc.)
- Lazy modifier (`*?`, `+?`, `??`) in regexes.
- Terminal priorities, templates, %declares, %import (except %import common).


**Terminals vs Rules & Greedy Lexing** 

| Concept          | Take-away                                                                    |
|------------------|------------------------------------------------------------------------------|
| Terminals (UPPER)| Matched first by the lexer – longest match wins.                             |
| Rules (lower)    | Combine terminals; cannot influence how text is tokenised.                   |
| Greedy lexer     | Never try to “shape” free text across multiple terminals – you’ll lose control. |

**Correct vs Incorrect Pattern Design** 

✅ **One bounded terminal handles free‑text between anchors**  
```
start: SENTENCE
SENTENCE: /[A-Za-z, ]*(the hero|a dragon)[A-Za-z, ]*(fought|saved)[A-Za-z, ]*(a treasure|the kingdom)[A-Za-z, ]*\./
```
❌ **Don’t split free‑text across multiple terminals/rules**  
```
start: sentence
sentence: /[A-Za-z, ]+/ subject /[A-Za-z, ]+/ verb /[A-Za-z, ]+/ object /[A-Za-z, ]+/
```

### 3.3 Example - SQL Dialect — MS SQL vs PostgreSQL

The following code example is now the canonical reference for building multi‑dialect SQL tools with CFGs. It demonstrates:

- Two isolated grammar definitions (`mssql_grammar_definition`, `postgres_grammar_definition`) encoding TOP vs LIMIT semantics.
- How to prompt, invoke, and inspect tool calls in a single script.
- A side‑by‑side inspection of the assistant’s responses.

Define the LARK grammars for different SQL dialects

In [10]:
import textwrap

# ----------------- grammars for MS SQL dialect -----------------
mssql_grammar = textwrap.dedent(r"""
            // ---------- Punctuation & operators ----------
            SP: " "
            COMMA: ","
            GT: ">"
            EQ: "="
            SEMI: ";"

            // ---------- Start ----------
            start: "SELECT" SP "TOP" SP NUMBER SP select_list SP "FROM" SP table SP "WHERE" SP amount_filter SP "AND" SP date_filter SP "ORDER" SP "BY" SP sort_cols SEMI

            // ---------- Projections ----------
            select_list: column (COMMA SP column)*
            column: IDENTIFIER

            // ---------- Tables ----------
            table: IDENTIFIER

            // ---------- Filters ----------
            amount_filter: "total_amount" SP GT SP NUMBER
            date_filter: "order_date" SP GT SP DATE

            // ---------- Sorting ----------
            sort_cols: "order_date" SP "DESC"

            // ---------- Terminals ----------
            IDENTIFIER: /[A-Za-z_][A-Za-z0-9_]*/
            NUMBER: /[0-9]+/
            DATE: /'[0-9]{4}-[0-9]{2}-[0-9]{2}'/
    """)

# ----------------- grammars for PostgreSQL dialect -----------------
postgres_grammar = textwrap.dedent(r"""
            // ---------- Punctuation & operators ----------
            SP: " "
            COMMA: ","
            GT: ">"
            EQ: "="
            SEMI: ";"

            // ---------- Start ----------
            start: "SELECT" SP select_list SP "FROM" SP table SP "WHERE" SP amount_filter SP "AND" SP date_filter SP "ORDER" SP "BY" SP sort_cols SP "LIMIT" SP NUMBER SEMI

            // ---------- Projections ----------
            select_list: column (COMMA SP column)*
            column: IDENTIFIER

            // ---------- Tables ----------
            table: IDENTIFIER

            // ---------- Filters ----------
            amount_filter: "total_amount" SP GT SP NUMBER
            date_filter: "order_date" SP GT SP DATE

            // ---------- Sorting ----------
            sort_cols: "order_date" SP "DESC"

            // ---------- Terminals ----------
            IDENTIFIER: /[A-Za-z_][A-Za-z0-9_]*/
            NUMBER: /[0-9]+/
            DATE: /'[0-9]{4}-[0-9]{2}-[0-9]{2}'/
    """)

### 3.4 Generate specific SQL dialect 
Let's define the prompt, and call the function to produce MS SQL dialect 

In [11]:
from openai import OpenAI
client = OpenAI()

sql_prompt_mssql = (
    "Call the mssql_grammar to generate a query for Microsoft SQL Server that retrieve the "
    "five most recent orders per customer, showing customer_id, order_id, order_date, and total_amount, "
    "where total_amount > 500 and order_date is after '2025-01-01'. "
)

response_mssql = client.responses.create(
    model="gpt-5.6-sol",
    input=sql_prompt_mssql,
    text={"format": {"type": "text"}},
    tools=[
        {
            "type": "custom",
            "name": "mssql_grammar",
            "description": "Executes read-only Microsoft SQL Server queries limited to SELECT statements with TOP and basic WHERE/ORDER BY. YOU MUST REASON HEAVILY ABOUT THE QUERY AND MAKE SURE IT OBEYS THE GRAMMAR.",
            "format": {
                "type": "grammar",
                "syntax": "lark",
                "definition": mssql_grammar
            }
        },
    ],
    parallel_tool_calls=False
)

print("--- MS SQL Query ---")
print(response_mssql.output[1].input)

--- MS SQL Query ---
SELECT TOP 5 customer_id, order_id, order_date, total_amount FROM orders WHERE total_amount > 500 AND order_date > '2025-01-01' ORDER BY order_date DESC;


The output SQL accurately uses "SELECT TOP" construct. 

In [12]:
sql_prompt_pg = (
    "Call the postgres_grammar to generate a query for PostgreSQL that retrieve the "
    "five most recent orders per customer, showing customer_id, order_id, order_date, and total_amount, "
    "where total_amount > 500 and order_date is after '2025-01-01'. "
)

response_pg = client.responses.create(
    model="gpt-5.6-sol",
    input=sql_prompt_pg,
    text={"format": {"type": "text"}},
    tools=[
        {
            "type": "custom",
            "name": "postgres_grammar",
            "description": "Executes read-only PostgreSQL queries limited to SELECT statements with LIMIT and basic WHERE/ORDER BY. YOU MUST REASON HEAVILY ABOUT THE QUERY AND MAKE SURE IT OBEYS THE GRAMMAR.",
            "format": {
                "type": "grammar",
                "syntax": "lark",
                "definition": postgres_grammar
            }
        },
    ],
    parallel_tool_calls=False,
)

print("--- PG SQL Query ---")
print(response_pg.output[1].input)

--- PG SQL Query ---
SELECT customer_id, order_id, order_date, total_amount FROM orders WHERE total_amount > 500 AND order_date > '2025-01-01' ORDER BY order_date DESC LIMIT 5;


Output highlights the same logical query - different physical syntax. Supply distinct grammars so the model can only produce valid statements for the chosen dialect.

| Dialect       | Generated Query                                              | Key Difference                          |
|---------------|--------------------------------------------------------------|------------------------------------------|
| MS SQL Server | SELECT TOP 5 customer_id, … ORDER BY order_date DESC;         | Uses `TOP N` clause before column list.  |
| PostgreSQL    | SELECT customer_id, … ORDER BY order_date DESC LIMIT 5;       | Uses `LIMIT N` after `ORDER BY`.         |



### 3.5 Example - Regex CFG Syntax

The following code example demonstrates using the Regex CFG syntax to constrain the freeform tool call to a certain timestamp pattern.

In [13]:
from openai import OpenAI
client = OpenAI()

timestamp_grammar_definition = r"^\d{4}-(0[1-9]|1[0-2])-(0[1-9]|[12]\d|3[01]) (?:[01]\d|2[0-3]):[0-5]\d$"

timestamp_prompt = (
        "Call the timestamp_grammar to save a timestamp for August 7th 2025 at 10AM."
)

response_mssql = client.responses.create(
    model="gpt-5.6-sol",
    input=timestamp_prompt,
    text={"format": {"type": "text"}},
    tools=[
        {
            "type": "custom",
            "name": "timestamp_grammar",
            "description": "Saves a timestamp in date + time in 24-hr format.",
            "format": {
                "type": "grammar",
                "syntax": "regex",
                "definition": timestamp_grammar_definition
            }
        },
    ],
    parallel_tool_calls=False
)

print("--- Timestamp ---")
print(response_mssql.output[1].input)

--- Timestamp ---
2025-08-07 10:00


### 3.5 Best Practices

Lark grammars can be tricky to perfect. While simple grammars perform most reliably, complex grammars often require iteration on the grammar definition itself, the prompt, and the tool description to ensure that the model does not go out of distribution.

- Keep terminals bounded – use `/[^.\n]{0,10}*\./` rather than `/.*\./`. Limit matches both by content (negated character class) and by length (`{M,N}` quantifier). 
- Prefer explicit char‑classes over `.` wildcards.
- Thread whitespace explicitly, e.g. using `SP = " "`, instead of a global `%ignore`.
- Describe your tool: tell the model exactly what the CFG accepts and instruct it to reason heavily about compliance.

**Troubleshooting**
- API rejects the grammar because it is too complex ➜ Simplify rules and terminals, remove `%ignore.*`.
- Unexpected tokens ➜ Confirm terminals aren’t overlapping; check greedy lexer.
- When the model drifts "out‑of‑distribution" (shows up as the model producing excessively long or repetitive outputs, it is syntactically valid but is semantically wrong):
    - Tighten the grammar.
    - Iterate on the prompt (add few-shot examples) and tool description (explain the grammar and instruct the model to reason to conform to it).
    - Experiment with a higher reasoning effort (e.g, bump from medium to high).

**Resources:**  
- Lark Docs – https://lark-parser.readthedocs.io/en/stable/
- Lark IDE – https://www.lark-parser.org/ide/
- LLGuidance Syntax – https://github.com/guidance-ai/llguidance/blob/main/docs/syntax.md
- Regex (Rust crate) – https://docs.rs/regex/latest/regex/#syntax

### 3.6 Takeaways 

Context-Free Grammar (CFG) support in GPT-5 lets you strictly constrain model output to match predefined syntax, ensuring only valid strings are generated. This is especially useful for enforcing programming language rules or custom formats, reducing post-processing and errors. By providing a precise grammar and clear tool description, you can make the model reliably stay within your target output structure.

## 4. Low Reasoning (Fastest Path)

### 4.1 Overview

For latency-sensitive tasks, use **`reasoning.effort: "low"`**. The model outputs very few reasoning tokens, minimising time-to-first-token. This is designed for deterministic, lightweight use cases: classification, extraction, short rewrites. The default effort is `medium` if omitted.

> **Note:** `minimal` effort existed in earlier GPT-5.x releases but is not available on any GPT-5.6 tier (`gpt-5.6-sol`/`terra`/`luna`) — nor was it available in `gpt-5.5`. `none` is now the fastest supported level on GPT-5.6, not `low` -- confirmed live: `reasoning={"effort": "none"}` runs successfully on `gpt-5.6-sol`.

In [14]:
from openai import OpenAI

client = OpenAI()

prompt = "Classify sentiment of the review as positive|neutral|negative. Return one word only." 


response = client.responses.create(
    model="gpt-5.6-sol",
    input= [{ 'role': 'developer', 'content': prompt }, 
            { 'role': 'user', 'content': 'The food that the restaurant was great! I recommend it to everyone.' }],
    reasoning = {
        "effort": "low"
    },
)

# Extract model's text output
output_text = ""
for item in response.output:
    if hasattr(item, "content"):
        for content in item.content:
            if hasattr(content, "text"):
                output_text += content.text

# Token usage details
usage = response.usage

print("--------------------------------")
print("Output:")
print(output_text)

--------------------------------
Output:
positive


### 4.2 Takeaways

Low reasoning effort runs GPT-5.6 with very few reasoning tokens to minimise latency and speed up time-to-first-token. Use it for deterministic, lightweight tasks (extraction, formatting, short rewrites, simple classification) where deep deliberation isn't needed. If you don't specify effort, it defaults to `medium` — set `low` explicitly when you want speed over thoroughness.

## 5. Hosted Shell Tool

The Responses API ships a **hosted shell tool** — OpenAI runs the commands in a managed Debian 12 container, so you don't have to stand up your own sandbox. The model issues shell commands, OpenAI executes them, and the output (including **images and files**, not just text) is returned as tool output.

In [15]:
# Hosted shell tool: OpenAI executes commands in a managed container.
response = client.responses.create(
    model="gpt-5.6-sol",
    input="List the Python files in the current directory and count their lines.",
    tools=[{"type": "shell"}],
    reasoning={"effort": "medium"},
)
if response.output_text:
    print(response.output_text)
else:
    # NOTE: as of this course's testing, the turn can end on the shell_call item
    # itself (status "completed") with no follow-up assistant message, so
    # output_text is empty even though the call succeeded. Showing the call here
    # instead of printing nothing -- see the markdown note below.
    for item in response.output:
        if item.type == "shell_call":
            print(f"[shell_call] {item.action.commands} (status: {item.status})")

[shell_call] ["find . -maxdepth 1 -type f -name '*.py' -print0 | sort -z | xargs -0 -r wc -l"] (status: completed)


## 6. Skills

**Skills** bundle a `SKILL.md` manifest (instructions + assets) that the model loads on demand inside the hosted shell container. Instead of stuffing every capability into one giant system prompt, define it as a skill and the model pulls it in when needed.

Skills attach via the shell tool's `environment.skills` list — upload your SKILL.md bundle first (`POST /v1/skills`), then reference the returned `skill_id`:

```python
tools=[{
    "type": "shell",
    "environment": {
        "type": "container_auto",
        "skills": [{"skill_id": "your-skill-id"}]
    }
}]
```

In [16]:
try:
    # Skills attach inside the shell tool's environment, not as a separate tool type.
    # Upload your SKILL.md bundle via POST /v1/skills to get a skill_id, then reference it here.
    response = client.responses.create(
        model="gpt-5.6-sol",
        input="Generate a quarterly revenue chart from this CSV and explain the trend.",
        tools=[{
            "type": "shell",
            "environment": {
                "type": "container_auto",
                "skills": [{"skill_id": "your-skill-id", "type": "skill_reference"}],  # replace with your actual skill_id
            }
        }],
        reasoning={"effort": "medium"},
    )
    print(response.output_text)
except Exception as e:
    print("Note: Replace 'your-skill-id' with a real skill ID obtained via POST /v1/skills")
    print(f"Error: {e}")


Note: Replace 'your-skill-id' with a real skill ID obtained via POST /v1/skills
Error: Error code: 404 - {'error': {'message': "System skill 'your-skill-id' not found.", 'type': 'invalid_request_error', 'param': None, 'code': None}}


## 7. Connectors

**Connectors** are OpenAI-maintained **MCP wrappers** for third-party services (Google apps, Dropbox, etc.). They let the model read/act on external data without you hand-writing each integration — OpenAI maintains the MCP server.

Pass an `mcp` tool with a `connector_id` (e.g. `connector_dropbox`) and your authorization token/connection id -- there is no separate `connector` tool type.

In [17]:
try:
    # Connectors are OpenAI-maintained MCP wrappers -- they use tool type "mcp"
    # with a connector_id, not a separate "connector" type.
    response = client.responses.create(
        model="gpt-5.6-sol",
        input="Find my most recent invoice in Dropbox and summarize the line items.",
        tools=[
            {
                "type": "mcp",
                "server_label": "dropbox",
                "connector_id": "connector_dropbox",
                # "authorization": "<your-oauth-token-or-connection-id>",  # required in production
            }
        ],
        reasoning={"effort": "medium"},
    )
    print(response.output_text)
except Exception as e:
    print("Note: requires an authorized OAuth connection (the 'authorization' field above).")
    print(f"Error: {e}")


Note: requires an authorized OAuth connection (the 'authorization' field above).
Error: Error code: 400 - {'error': {'message': "Must specify 'authorization' parameter with 'connector_id'.", 'type': 'invalid_request_error', 'param': 'authorization', 'code': None}}
